# 실패 분석(Failure Analysis)

## 학습 목표

- agent 시스템에서 자주 나오는 failure mode를 이해한다.
- 재사용 가능한 failure taxonomy와 severity 정보를 읽는 법을 익힌다.
- failure summary를 trace와 실제 실행 예시로 연결한다.
- 관찰된 실패를 다음 개선 과제로 우선순위화한다.


## 개념 설명

다른 노트북과 마찬가지로, 여기서도 먼저 runtime을 확인한다. failure analysis는 어떤 환경에서 생성된 artifact를 보고 있는지 알아야 해석이 가능하다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Agent 시스템의 failure mode

agent 시스템은 한 군데에서만 실패하지 않는다. retrieval이 근거를 놓칠 수도 있고, planner가 너무 얕은 계획을 고를 수도 있으며, synthesis가 중요한 정보를 누락할 수도 있다. 또 fallback policy가 답해야 할 때 abstain하거나, 반대로 abstain해야 할 때 답할 수도 있다. taxonomy가 필요한 이유는 이런 실패를 이름 붙여 논의 가능하게 만들기 위해서다.


## 구현

이 노트북은 failure logic을 inline으로 다시 쓰지 않고, 재사용 가능한 taxonomy와 analyzer 모듈을 그대로 가져와 쓴다. 그래야 notebook에서 보는 failure 분석과 실제 evaluation 코드가 같은 기준을 공유하게 된다.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import get_paths
from src.evaluator import attach_failure_improvements, extract_failure_cases, run_evaluation_suite
from src.failure_analyzer import analyze_failures, generate_failure_report, taxonomy_frame
from src.trace_debug import display_trace
from src.utils import read_json

paths = get_paths()
results_path = paths.eval_dir / 'eval_results.json'
if results_path.exists():
    results = pd.DataFrame(read_json(results_path))
else:
    results, _ = run_evaluation_suite(repeats=2, persist_outputs=True)

results.head(5)

## 실패 taxonomy

아래 taxonomy는 각 failure를 더 운영적으로 읽을 수 있게 만든다. stage는 어디를 봐야 하는지 알려주고, severity는 얼마나 급한 문제인지 말해주며, mitigation은 다음 개선 가설을 제시한다.


In [ ]:
taxonomy = taxonomy_frame().sort_values(['severity', 'stage', 'failure_type']).reset_index(drop=True)
severity_colors = {'critical': '#FEE2E2', 'major': '#FEF3C7', 'minor': '#DBEAFE'}

def color_row(row):
    color = severity_colors.get(row['severity'], '#FFFFFF')
    return [f'background-color: {color}' for _ in row]

taxonomy.style.apply(color_row, axis=1)

## 실패 케이스 추출(Extract failure cases)

이제 전체 결과에서 실패한 run만 골라내고, 개선 아이디어를 함께 붙인다. 이런 narrowing 과정이 있어야 실제 연구 흐름처럼 전체 실험에서 문제 사례만 따로 떼어 분석할 수 있다.


In [ ]:
failures = attach_failure_improvements(extract_failure_cases(results))
failures[['system', 'question_id', 'expected_question_type', 'failure_type', 'improvement_idea']].head(12)

## trace 점검(Trace inspection)

집계 수치도 중요하지만, 개별 trace는 실패 메커니즘을 훨씬 선명하게 보여준다. 아래 셀에서는 agent workflow의 대표 실패 사례 몇 개를 골라서, 어느 node에서 문제가 시작됐는지 직접 확인한다.


In [ ]:
from pathlib import Path

analysis = analyze_failures(failures)
report_path = paths.reports_dir / 'failure_report.md'
generate_failure_report(analysis, report_path)

agent_failures = failures[failures['system'] == 'agent_workflow'].head(3)
trace_previews = []
for _, row in agent_failures.iterrows():
    trace_path = paths.traces_dir / f"{row['question_id']}_run{int(row['run_id'])}.json"
    trace = read_json(trace_path)['trace'] if trace_path.exists() else []
    trace_previews.append(
        {
            'question_id': row['question_id'],
            'failure_type': row['failure_type'],
            'trace_path': str(trace_path),
            'trace_steps': len(trace),
        }
    )

trace_preview_frame = pd.DataFrame(trace_previews)
display(trace_preview_frame)
for preview in trace_previews:
    print(f"Trace for {preview['question_id']} ({preview['failure_type']})")
    display_trace(read_json(Path(preview['trace_path']))['trace'])

## 실험

failure에 이름을 붙이고 나면, 이제는 어느 stage에 몰리는지 묻는 것이 중요하다. stage와 severity 분포를 보면 다음 iteration에서 retrieval을 먼저 손볼지, planning을 손볼지, synthesis를 손볼지, fallback policy를 조정할지 판단하기 쉬워진다.


In [ ]:
stage_distribution = pd.Series(analysis['stage_distribution']).sort_values(ascending=False)
severity_distribution = pd.Series(analysis['severity_distribution']).sort_values(ascending=False)
improvement_actions = pd.DataFrame(analysis['top_improvement_actions'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stage_distribution.plot(kind='bar', color='#4C78A8', ax=axes[0], title='Failures by Stage')
axes[0].set_xlabel('stage')
axes[0].set_ylabel('count')
severity_distribution.plot(kind='pie', autopct='%1.0f%%', ax=axes[1], title='Failure Severity Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

display(improvement_actions)

## 결과 해석

failure taxonomy의 가치는 증상을 바로 개입 지점으로 연결해준다는 데 있다. "workflow가 약해 보인다" 수준에서 멈추지 않고, "retrieval noise가 많다" 혹은 "critical issue가 fallback에 몰려 있다"처럼 더 구체적인 개선 문장으로 바꿀 수 있다.


In [ ]:
pd.Series({
    'total_failures': int(analysis['total_failure_instances']),
    'unique_failure_types': len(analysis['failure_distribution']),
    'report_path': str(report_path),
    'top_improvement_action': improvement_actions.iloc[0]['mitigation'] if not improvement_actions.empty else 'None',
})

## 핵심 정리

- 이 실험을 통해 failure analysis는 나쁜 결과를 숨기는 작업이 아니라, **다음 개선을 설계하는 근거 생산 과정**이라는 점을 확인했다.
- stage와 severity 정보가 있으면 anecdote가 아니라 우선순위 있는 개선 과제로 정리할 수 있다.
- trace를 같이 봐야 summary label이 실제 workflow 동작과 연결된다.
- 면접에서는 "실패를 어떻게 다뤘는가"라는 질문에 대해, **taxonomy, trace, improvement action** 세 단계를 엮어서 설명하면 강하다.
